In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.select import Select

download = 0
begin_search_download = 0
t_end = 0
redo_counter = 0
download_row = 0
rows_in_table = 0
failed_file = open("failed_structure_downloads.txt", mode="a", encoding="utf-8")
fractional_file = open("rruff_files.txt", mode="a", encoding="utf-8")

compound_list = open("RRUFF_compound_names.txt")

compound_lines = compound_list.readlines()

compound_list.close()

compound_list = compound_lines[0].split(",")

clean_compound_list = []

for compound in compound_list:
	compound = compound.split("'")[1]
	clean_compound_list.append(compound)

clean_compound_list = list(set(clean_compound_list))


def rename_most_recently_modified_file(directory, new_name):
    # Ensure the provided directory exists
    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist.")
        return

    # Initialize variables to store the most recent file info
    most_recent_file = None
    most_recent_mtime = 0

    # Iterate over all files in the directory
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)

        # Skip directories
        if os.path.isdir(file_path):
            continue

        # Get the last modification time of the file
        mtime = os.path.getmtime(file_path)

        # Update the most recent file info if this file is more recent
        if mtime > most_recent_mtime:
            most_recent_file = file_path
            most_recent_mtime = mtime

    # If a file was found, rename it
    if most_recent_file:
        new_file_path = os.path.join(directory, new_name)
        os.rename(most_recent_file, new_file_path)
        print(f"Renamed '{most_recent_file}' to '{new_file_path}'")
    else:
        print("No files found in the directory.")

s = Service("./chromedriver.exe")
driver = webdriver.Chrome(service = s)
#driver.get('https://icsd.fiz-karlsruhe.de/search/basic.xhtml')
#driver.get('https://icsd.products.fiz-karlsruhe.de/')
for compound in clean_compound_list:
	driver.get('https://rruff.geo.arizona.edu/AMS/minerals/' + compound)

	time.sleep(1)

	try:
		CIF_buttons = driver.find_elements(by=By.LINK_TEXT, value="Download CIF data")
        counter = 0 
		for CIF_button in CIF_buttons:
            counter += 1
			CIF_button.click()
            rename_most_recently_modified_file(dirname, compound + "cif" + str(counter))
			time.sleep(1)


		diffraction_buttons = driver.find_elements(by=By.LINK_TEXT, value="Download diffraction data")
		for diffraction_button in diffraction_buttons:
			diffraction_button.click()
            counter += 1
            rename_most_recently_modified_file(dirname, compound + "cif" + str(counter))
			time.sleep(1)

		fractional_file.write(compound + "\n")

	except:
		failed_file.write(compound + "\n")


fractional_file.close()
failed_file.close()